In [76]:
import cv2
import json
import random
import tkinter as tk
from pathlib import Path

In [77]:
#AQCUIRE SCREEN SIZE

def get_screen_size():
    root = tk.Tk()
    root.withdraw()
    w, h = root.winfo_screenwidth(), root.winfo_screenheight()
    root.destroy()
    return w, h

In [78]:
#SHOW STATS WHILE SELECTING

def draw_text_block(img, lines, origin=(15, 15)):
    """Draw a solid dark background box behind each line of text for reliable contrast."""
    x0, y0 = origin
    line_h = 26
    pad = 6
    max_w = max(cv2.getTextSize(line, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)[0][0] for line in lines)

    cv2.rectangle(
        img,
        (x0 - pad, y0 - pad),
        (x0 + max_w + pad, y0 + len(lines) * line_h + pad),
        (0, 0, 0),
        -1,
    )
    for i, line in enumerate(lines):
        y = y0 + (i + 1) * line_h - 8
        cv2.putText(img, line, (x0, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

In [79]:
#RESULTS STORAGE

def build_json_range(hs, ss, vs, colour_name, margin_s=20, margin_v=20, margin_h=2):
    if colour_name == "red":
        low_hs = [hh for hh in hs if hh <= 90]
        high_hs = [hh for hh in hs if hh > 90]
        ranges = []
        if low_hs:
            ranges.append([[0, max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)],
                           [min(10, max(low_hs) + margin_h), 255, 255]])
        if high_hs:
            ranges.append([[max(170, min(high_hs) - margin_h), max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)],
                           [180, 255, 255]])
        if not ranges:
            ranges = [[[0, max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)], [10, 255, 255]],
                      [[170, max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)], [180, 255, 255]]]
        return ranges
    else:
        return [[[max(0, min(hs) - margin_h), max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)],
                 [max(hs) + margin_h, 255, 255]]]

In [80]:
#PRINT RESULTS

def print_summary(samples_by_colour):
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    result = {}
    for colour in ("red", "green", "blue"):
        samples = samples_by_colour[colour]
        if not samples:
            print(f"\n{colour.upper()}: no samples taken, skipped.")
            continue

        hs = [s["hsv"][0] for s in samples]
        ss = [s["hsv"][1] for s in samples]
        vs = [s["hsv"][2] for s in samples]
        avg_h, avg_s, avg_v = sum(hs) / len(hs), sum(ss) / len(ss), sum(vs) / len(vs)
        json_range = build_json_range(hs, ss, vs, colour)
        result[colour] = json_range

        print(f"\n{colour.upper()}: {len(samples)} sample(s)")
        print(f"  Average HSV: ({avg_h:.1f}, {avg_s:.1f}, {avg_v:.1f})")
        print(f"  Accepted range: {json_range}")
    return result

def print_all_samples(samples_by_colour):
    print("\n" + "=" * 60)
    print("ALL SAMPLES")
    print("=" * 60)
    for colour in ("red", "green", "blue"):
        samples = samples_by_colour[colour]
        if not samples:
            continue
        print(f"\n--- {colour.upper()} ({len(samples)} samples) ---")
        for i, s in enumerate(samples, start=1):
            print(f"  {i}. frame {s['frame']:>5}   coord ({s['x']},{s['y']})   HSV {s['hsv']}")

In [81]:
class ColourCalibrator:
    """
    Interactive HSV colour calibration tool.
    Press 'r', 'g', or 'b' to select which dot colour you're sampling.
    Hover to preview HSV/coords live. Click to sample the currently
    selected colour on the currently shown frame, then a new random
    frame loads automatically. Switch colours anytime by pressing a
    different key. Press 'q' when done to save and see a full summary.
    """

    KEY_TO_COLOUR = {ord("r"): "red", ord("g"): "green", ord("b"): "blue"}
    WINDOW_NAME = "Colour calibration"

    def __init__(self, video_path, output_dir="colour calibration", margin=0.9):
        self.video_path = FOOTAGE_DIR / video_path
        if not self.video_path.exists():
            raise FileNotFoundError(f"Video file not found: {self.video_path}")

        self.output_dir = Path(output_dir)
        self.margin = margin

        self.cap = cv2.VideoCapture(str(self.video_path))
        self.total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if self.total_frames <= 0:
            self.cap.release()
            raise RuntimeError(f"Could not determine frame count for {self.video_path}")

        self.screen_w, self.screen_h = get_screen_size()

        self.colour = None
        self.samples = {"red": [], "green": [], "blue": []}
        self.hsv = None
        self.scale = 1.0
        self.frame_number = None
        self.base_display = None
        self.hover = None

        cv2.namedWindow(self.WINDOW_NAME, cv2.WINDOW_NORMAL)
        cv2.setMouseCallback(self.WINDOW_NAME, self.on_mouse)

    def load_random_frame(self):
        frame_number = random.randint(0, self.total_frames - 1)
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = self.cap.read()
        if not ret:
            return self.load_random_frame()

        h, w = frame.shape[:2]
        scale = min((self.screen_w * self.margin) / w, (self.screen_h * self.margin) / h, 1.0)
        disp_w, disp_h = int(w * scale), int(h * scale)
        display_frame = cv2.resize(frame, (disp_w, disp_h)) if scale != 1.0 else frame.copy()

        self.hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        self.scale = scale
        self.frame_number = frame_number
        self.base_display = display_frame
        self.hover = None

        cv2.resizeWindow(self.WINDOW_NAME, disp_w, disp_h)
        self.redraw()

    def redraw(self):
        display_frame = self.base_display.copy()

        lines = [
            "r = red   g = green   b = blue   q = quit & save",
            f"Frame: {self.frame_number} / {self.total_frames}",
            f"Sampling: {self.colour.upper() if self.colour else '(press r/g/b)'}"
            f"   red:{len(self.samples['red'])}  green:{len(self.samples['green'])}  blue:{len(self.samples['blue'])}",
        ]
        draw_text_block(display_frame, lines, origin=(15, 15))

        if self.hover is not None:
            hx, hy, hh, hs_, hv = self.hover
            draw_text_block(display_frame, [f"({hx},{hy})  HSV: ({hh},{hs_},{hv})"],
                             origin=(15, display_frame.shape[0] - 45))

        cv2.imshow(self.WINDOW_NAME, display_frame)
        try:
            cv2.setWindowProperty(self.WINDOW_NAME, cv2.WND_PROP_TOPMOST, 1)
        except cv2.error:
            pass

    def on_mouse(self, event, x, y, flags, param):
        if event == cv2.EVENT_MOUSEMOVE:
            h, w = self.hsv.shape[:2]
            orig_x, orig_y = int(x / self.scale), int(y / self.scale)
            if 0 <= orig_x < w and 0 <= orig_y < h:
                hh, ss, vv = self.hsv[orig_y, orig_x]
                self.hover = (orig_x, orig_y, int(hh), int(ss), int(vv))
                self.redraw()
        elif event == cv2.EVENT_LBUTTONDOWN:
            if self.colour is None:
                return
            orig_x, orig_y = int(x / self.scale), int(y / self.scale)
            hh, ss, vv = self.hsv[orig_y, orig_x]
            self.samples[self.colour].append({
                "colour": self.colour,
                "frame": self.frame_number,
                "x": orig_x,
                "y": orig_y,
                "hsv": (int(hh), int(ss), int(vv)),
            })
            self.load_random_frame()

    def run(self):
        self.load_random_frame()
        while True:
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key in self.KEY_TO_COLOUR:
                self.colour = self.KEY_TO_COLOUR[key]
                self.redraw()

        self.cap.release()
        cv2.destroyAllWindows()

        result = print_summary(self.samples)
        print_all_samples(self.samples)

        self.output_dir.mkdir(parents=True, exist_ok=True)
        out_path = self.output_dir / f"{self.video_path.stem}_colour_calibration.json"
        out_path.write_text(json.dumps(result, indent=2))
        print(f"\nSaved colour ranges to {out_path}")

        return result

def calibrate_colours(video_path, output_dir="colour_calibration"):
    """Entry point: runs the interactive calibration tool for a video."""
    calibrator = ColourCalibrator(video_path, output_dir=output_dir)
    return calibrator.run()

In [82]:
FOOTAGE_DIR = Path("footage")
if __name__ == "__main__":
    calibrate_colours("Test_1.MOV")


SUMMARY

RED: 10 sample(s)
  Average HSV: (7.5, 249.6, 151.9)
  Accepted range: [[[0, 210, 127], [10, 255, 255]]]

GREEN: 10 sample(s)
  Average HSV: (38.8, 131.0, 110.5)
  Accepted range: [[[24, 32, 55], [49, 255, 255]]]

BLUE: 5 sample(s)
  Average HSV: (128.4, 218.2, 90.2)
  Accepted range: [[[126, 151, 57], [131, 255, 255]]]

ALL SAMPLES

--- RED (10 samples) ---
  1. frame   405   coord (1136,637)   HSV (8, 255, 156)
  2. frame   452   coord (1193,608)   HSV (8, 255, 147)
  3. frame    24   coord (752,38)   HSV (6, 230, 155)
  4. frame   198   coord (981,672)   HSV (8, 255, 156)
  5. frame   460   coord (1004,670)   HSV (8, 255, 151)
  6. frame     8   coord (751,44)   HSV (7, 241, 151)
  7. frame   463   coord (908,654)   HSV (7, 252, 153)
  8. frame   412   coord (1008,669)   HSV (8, 248, 154)
  9. frame   290   coord (1169,620)   HSV (7, 250, 149)
  10. frame   530   coord (906,656)   HSV (8, 255, 147)

--- GREEN (10 samples) ---
  1. frame   313   coord (622,783)   HSV (26, 7